# Return Statistics and Historical Volatility

This notebook connects daily stock-price movement to the volatility input used later in the options work.

The important idea is not just that volatility can be calculated from historical returns. The important idea is that volatility summarizes uncertainty in the underlying stock, and that uncertainty is one of the main reasons options have value.


## Price Data

The notebook uses a local sample price file by default so the analysis runs without internet access. For the actual project, this can be switched to a live ticker download through `yfinance` by changing the configuration cell below.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_prices
from src.statistics import (
    add_return_columns,
    annualized_volatility,
    normal_pdf,
    return_summary,
)

figures_dir = PROJECT_ROOT / "figures"
processed_dir = PROJECT_ROOT / "data" / "processed"

figures_dir.mkdir(exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
USE_YFINANCE = False

ticker = "SPY"
start_date = "2024-01-01"
end_date = "2025-01-01"

if USE_YFINANCE:
    prices = load_prices(
        ticker=ticker,
        start=start_date,
        end=end_date,
        use_yfinance=True,
    )
else:
    prices = load_prices(
        csv_path=PROJECT_ROOT / "data" / "raw" / "sample_prices.csv",
        ticker="SPY_SAMPLE",
    )

prices.head()


## Daily Returns

Simple returns measure the percentage change in price from one trading day to the next. Log returns are also included because they are often more convenient in finance, especially when returns are compounded across time.


In [ ]:
returns_table = add_return_columns(prices)

returns_table.to_csv(processed_dir / "returns_table.csv", index=False)

returns_table.head(10)


## Volatility Estimate

Historical volatility is estimated from the standard deviation of daily returns. To express daily volatility as an annual number, the daily estimate is multiplied by the square root of 252, which is the standard approximation for the number of trading days in a year.


In [ ]:
summary = return_summary(returns_table["LogReturn"])
summary.to_csv(processed_dir / "volatility_summary.csv", index=False)

summary


In [ ]:
vol = annualized_volatility(returns_table["LogReturn"])
print(f"Annualized volatility estimate: {vol:.2%}")


## Return Histogram

A return histogram shows the full distribution of daily price changes. This is more informative than looking only at the average return because the shape, spread, and tails of the distribution matter for risk.


In [ ]:
daily_returns = returns_table["LogReturn"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(daily_returns, bins=35, density=True, alpha=0.75)
plt.title("Daily Log Return Distribution")
plt.xlabel("Daily Log Return")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.tight_layout()

hist_path = figures_dir / "return_histogram.png"
plt.savefig(hist_path, dpi=160)
plt.show()

print(f"Saved: {hist_path}")


## Normal Comparison

The normal curve is a useful benchmark, but real returns often have features the normal distribution does not capture perfectly, especially heavier tails. For this project, the comparison is mainly useful because later pricing models use volatility as a core input, and volatility is estimated from the return distribution.


In [ ]:
mu = daily_returns.mean()
sigma = daily_returns.std(ddof=1)

x = np.linspace(daily_returns.min(), daily_returns.max(), 400)
y = normal_pdf(x, mean=mu, std=sigma)

plt.figure(figsize=(8, 5))
plt.hist(daily_returns, bins=35, density=True, alpha=0.75, label="Observed returns")
plt.plot(x, y, linewidth=2, label="Normal benchmark")
plt.title("Daily Log Returns vs Normal Benchmark")
plt.xlabel("Daily Log Return")
plt.ylabel("Density")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

comparison_path = figures_dir / "return_histogram_normal_comparison.png"
plt.savefig(comparison_path, dpi=160)
plt.show()

print(f"Saved: {comparison_path}")


## Why This Matters for Options

Volatility is central to option value because an option benefits from movement. A call benefits from large upward moves, while a put benefits from large downward moves. When volatility is higher, the range of possible future stock prices is wider, which increases the chance that the option finishes with meaningful payoff.

This is also why hedging error should be studied as a distribution. A hedge can perform differently across different price paths. Looking at one final error number hides the range of possible outcomes, including the tail cases that matter most for risk management.
